# residual-skip-add — ex2: compare identity vs 1×1-conv shortcut on matched and mismatched cases

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `residual-skip-add`. Running the final beacon cell reports progress against the `CNN: Residual skip-connection add` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Residual skip-connection add` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`residual-skip-add`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "residual-skip-add"
DD_SUBTOPIC = "CNN: Residual skip-connection add"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Identity vs 1×1-conv shortcut — what each branch produces

Ex1 BUILT the conditional ResidualBlock. The deepening move is to INSPECT both branches on a shape-mismatched case — show how the 1×1 shortcut rescues the add when channels (or stride) differ.

Two scenarios:

- **Matched (in=out, stride=1):** `self.skip = nn.Identity()`. Identity returns its input unchanged. `conv(x) + skip(x)` adds the fresh conv output to the ORIGINAL input directly. Zero extra params, zero FLOPs from the skip branch.
- **Mismatched (in≠out OR stride>1):** `self.skip = nn.Conv2d(in, out, kernel_size=1, stride=first_stride)`. The 1×1 conv changes channel count (and downsamples spatially via stride) without introducing receptive-field artifacts — every output pixel is a linear combo of the SAME input pixel across input channels.

**Why a 1×1 conv, not a Linear.** A spatial tensor `(B, C, H, W)` needs the same `H, W` to be summable with the conv branch's output. A 1×1 conv operates per-pixel and preserves `(H, W)` (modulo stride). `nn.Linear` would require flattening and lose the spatial axis.

**Why no bias on a typical 1×1 shortcut.** In real ResNet, the shortcut conv is followed by BatchNorm — the BN's `beta` absorbs any constant offset, so the conv's bias is redundant. This toy drill omits BN, so a default bias is fine.

### Exercise 2 — compare identity vs 1×1-conv shortcut on matched and mismatched cases

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the two ResidualBlock shortcut variants by reporting the skip-branch type, its parameter count, and the shape of `block.skip(x)` for both a matched (Identity) and a mismatched (1×1 Conv) configuration.
> Keywords: residual, shortcut, 1x1-conv, identity, shape-contrast
> ```

**KCs targeted:** `identity-shortcut-zero-params`, `1x1-shortcut-channel-projection`

Implement `ex2_analyze_shortcut(in_channels, out_channels, first_stride, H, W)`.

Build a ResidualBlock with the EXACT same conditional-shortcut logic as ex1:
- if `in_channels == out_channels and first_stride == 1` → `self.skip = nn.Identity()`
- else → `self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=first_stride, padding=0)`

Then run a `(2, in_channels, H, W)` random input through the skip branch and return:
```
{
  'skip_type': 'identity' | 'conv1x1',
  'skip_n_params': int,                # sum of numel over self.skip.parameters()
  'skip_out_shape': tuple,             # tuple(block.skip(x).shape)
  'conv_out_shape': tuple,             # tuple(block.conv(x).shape)
  'sum_shape': tuple,                  # tuple((conv(x) + skip(x)).shape)
}
```

Constraints:
1. The conv branch must be `nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=first_stride, padding=1)` (matches ex1).
2. `skip_type` is `'identity'` exactly when `self.skip` is `nn.Identity`; otherwise `'conv1x1'`.
3. Use `t.manual_seed(0)` BEFORE constructing the block AND before generating the input so the test is deterministic.
4. Tuples (not torch.Size) — wrap with `tuple(...)`.

In [ ]:
def ex2_analyze_shortcut(in_channels: int, out_channels: int,
                          first_stride: int, H: int, W: int) -> dict:
    """Build a ResidualBlock and report skip-branch shape + param counts."""
    raise NotImplementedError()


def _test_ex2():
    # === Matched case (in=out, stride=1) → Identity, 0 params, same shape ===
    rep = ex2_analyze_shortcut(in_channels=8, out_channels=8, first_stride=1, H=16, W=16)
    assert rep['skip_type'] == 'identity', f'in=out=8 stride=1 should be identity, got {rep["skip_type"]}'
    assert rep['skip_n_params'] == 0, f'identity must have 0 params, got {rep["skip_n_params"]}'
    assert rep['skip_out_shape'] == (2, 8, 16, 16), f'identity preserves shape, got {rep["skip_out_shape"]}'
    assert rep['conv_out_shape'] == (2, 8, 16, 16)
    assert rep['sum_shape'] == (2, 8, 16, 16)

    # === Channel-mismatch case (in=4 out=8, stride=1) → 1x1 conv ===
    rep = ex2_analyze_shortcut(in_channels=4, out_channels=8, first_stride=1, H=16, W=16)
    assert rep['skip_type'] == 'conv1x1', f'in!=out should be conv1x1, got {rep["skip_type"]}'
    # 1x1 conv from 4 -> 8 channels: weight (8, 4, 1, 1) = 32, bias (8,) = 8, total 40.
    assert rep['skip_n_params'] == 40, f'1x1 conv 4->8 has 8*4 + 8 = 40 params, got {rep["skip_n_params"]}'
    assert rep['skip_out_shape'] == (2, 8, 16, 16)
    assert rep['conv_out_shape'] == (2, 8, 16, 16)
    assert rep['sum_shape'] == (2, 8, 16, 16)

    # === Stride-mismatch case (in=8 out=8, stride=2) → 1x1 conv, halved spatial ===
    rep = ex2_analyze_shortcut(in_channels=8, out_channels=8, first_stride=2, H=16, W=16)
    assert rep['skip_type'] == 'conv1x1', f'stride=2 should force conv1x1, got {rep["skip_type"]}'
    # 1x1 conv 8->8: 8*8 + 8 = 72.
    assert rep['skip_n_params'] == 72, f'1x1 conv 8->8 has 72 params, got {rep["skip_n_params"]}'
    # 16 -> 8 via stride 2 (kernel=1, padding=0): (16 + 0 - 1)//2 + 1 = 8.
    assert rep['skip_out_shape'] == (2, 8, 8, 8), f'stride 2 halves spatial; got {rep["skip_out_shape"]}'
    # conv branch (3x3, stride=2, padding=1): (16 + 2 - 3)//2 + 1 = 8.
    assert rep['conv_out_shape'] == (2, 8, 8, 8)
    assert rep['sum_shape'] == (2, 8, 8, 8)

    # === Combined mismatch (in=4 out=16, stride=2) ===
    rep = ex2_analyze_shortcut(in_channels=4, out_channels=16, first_stride=2, H=8, W=8)
    assert rep['skip_type'] == 'conv1x1'
    # 4 -> 16 channels via 1x1: 16*4 + 16 = 80.
    assert rep['skip_n_params'] == 80
    # 8 spatial down to (8 - 1)//2 + 1 = 4.
    assert rep['skip_out_shape'] == (2, 16, 4, 4)
    assert rep['conv_out_shape'] == (2, 16, 4, 4)
    assert rep['sum_shape'] == (2, 16, 4, 4)

    # === Non-square spatial input still consistent ===
    rep = ex2_analyze_shortcut(in_channels=2, out_channels=2, first_stride=1, H=7, W=11)
    assert rep['skip_type'] == 'identity'
    assert rep['skip_out_shape'] == (2, 2, 7, 11)
    assert rep['sum_shape'] == (2, 2, 7, 11)

    # === All shape tuples are plain Python tuples, not torch.Size ===
    for k in ('skip_out_shape', 'conv_out_shape', 'sum_shape'):
        assert type(rep[k]) is tuple, f'{k} must be plain tuple, got {type(rep[k]).__name__}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_analyze_shortcut(in_channels, out_channels, first_stride, H, W):
    class ResidualBlock(nn.Module):
        def __init__(self, ic, oc, stride):
            super().__init__()
            self.conv = nn.Conv2d(ic, oc, kernel_size=3, stride=stride, padding=1)
            if ic == oc and stride == 1:
                self.skip = nn.Identity()
            else:
                self.skip = nn.Conv2d(ic, oc, kernel_size=1, stride=stride, padding=0)
        def forward(self, x):
            return self.conv(x) + self.skip(x)

    t.manual_seed(0)
    block = ResidualBlock(in_channels, out_channels, first_stride)
    t.manual_seed(0)
    x = t.randn(2, in_channels, H, W)
    skip_out = block.skip(x)
    conv_out = block.conv(x)
    sum_out = conv_out + skip_out
    return {
        'skip_type': 'identity' if isinstance(block.skip, nn.Identity) else 'conv1x1',
        'skip_n_params': sum(p.numel() for p in block.skip.parameters()),
        'skip_out_shape': tuple(skip_out.shape),
        'conv_out_shape': tuple(conv_out.shape),
        'sum_shape': tuple(sum_out.shape),
    }
```

**`nn.Identity()` has zero parameters.** Iterating `Identity().parameters()` yields nothing — so `sum(p.numel() for p in ...)` is 0. Useful as a free-of-charge fallback when you want a uniform `forward(x): conv(x) + skip(x)` interface but only sometimes need real work in the skip branch.

**1×1 conv params: `out_channels * in_channels + out_channels`.** Weight is `(out, in, 1, 1)` flattening to `out * in`; bias is `(out,)`. For 4→8 that's `8*4 + 8 = 40`. The bias is the default; in real ResNet you'd disable it because BN absorbs the shift.

**Why seed twice.** The block-construction seed picks the conv's initial weights; the input-construction seed picks the random input. Setting `manual_seed` before each one is more predictable than relying on one seed at the top and trusting the order of consumption inside `t.randn` calls.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()